In [18]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('database.sqlite')

In [19]:
master = pd.read_sql_query("SELECT * FROM sqlite_master", conn)
master

,type,name,tbl_name,rootpage,sql
0,table,Player,Player,2,CREATE TABLE [Player] (\n\t[Player_Id]\tintege...
1,table,Extra_Runs,Extra_Runs,3,CREATE TABLE [Extra_Runs] (\n\t[Match_Id]\tint...
2,index,sqlite_autoindex_Extra_Runs_1,Extra_Runs,4,None
3,table,Batsman_Scored,Batsman_Scored,7,CREATE TABLE [Batsman_Scored] (\n\t[Match_Id]\...
4,index,sqlite_autoindex_Batsman_Scored_1,Batsman_Scored,8,None
5,table,Batting_Style,Batting_Style,10,CREATE TABLE [Batting_Style] (\n\t[Batting_Id]...
6,table,Bowling_Style,Bowling_Style,11,CREATE TABLE [Bowling_Style] (\n\t[Bowling_Id]...
7,table,Country,Country,12,CREATE TABLE [Country] (\n\t[Country_Id]\tinte...
8,table,Season,Season,14,CREATE TABLE [Season] (\n\t[Season_Id]\tintege...
9,table,City,City,15,CREATE TABLE [City] (\n\t[City_Id]\tinteger NO...


In [20]:
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type = 'table'", conn)
tables

,name
0,Player
1,Extra_Runs
2,Batsman_Scored
3,Batting_Style
4,Bowling_Style
5,Country
6,Season
7,City
8,Outcome
9,Win_By


In [21]:
for name in tables['name']:
    df= pd.read_sql_query(f"SELECT * FROM {name} LIMIT 1", conn)
    print(name, list(df.columns))

Player ['Player_Id', 'Player_Name', 'DOB', 'Batting_hand', 'Bowling_skill', 'Country_Name']
Extra_Runs ['Match_Id', 'Over_Id', 'Ball_Id', 'Extra_Type_Id', 'Extra_Runs', 'Innings_No']
Batsman_Scored ['Match_Id', 'Over_Id', 'Ball_Id', 'Runs_Scored', 'Innings_No']
Batting_Style ['Batting_Id', 'Batting_hand']
Bowling_Style ['Bowling_Id', 'Bowling_skill']
Country ['Country_Id', 'Country_Name']
Season ['Season_Id', 'Man_of_the_Series', 'Orange_Cap', 'Purple_Cap', 'Season_Year']
City ['City_Id', 'City_Name', 'Country_id']
Outcome ['Outcome_Id', 'Outcome_Type']
Win_By ['Win_Id', 'Win_Type']
Wicket_Taken ['Match_Id', 'Over_Id', 'Ball_Id', 'Player_Out', 'Kind_Out', 'Fielders', 'Innings_No']
Venue ['Venue_Id', 'Venue_Name', 'City_Id']
Extra_Type ['Extra_Id', 'Extra_Name']
Out_Type ['Out_Id', 'Out_Name']
Toss_Decision ['Toss_Id', 'Toss_Name']
Umpire ['Umpire_Id', 'Umpire_Name', 'Umpire_Country']
Team ['Team_Id', 'Team_Name']
Ball_by_Ball ['Match_Id', 'Over_Id', 'Ball_Id', 'Innings_No', 'Team_Batti

In [22]:
player_match = pd.read_sql_query("SELECT * FROM Player_Match", conn)
player_match

,Match_Id,Player_Id,Role_Id,Team_Id
0,335987,1,1,1
1,335987,2,3,1
2,335987,3,3,1
3,335987,4,3,1
4,335987,5,3,1
...,...,...,...,...
12689,981024,385,3,11
12690,981024,394,3,11
12691,981024,429,3,11
12692,981024,434,3,2


In [23]:
q = """
SELECT b.Striker, s.Runs_Scored
FROM Ball_by_Ball b
JOIN Batsman_Scored s
  ON b.Match_ID = s.Match_ID
AND b.Over_ID = s.Over_ID
AND b.Ball_ID = s.Ball_ID
AND b.Innings_No = s.Innings_No
LIMIT 10
"""
pd.read_sql_query(q, conn)

,Striker,Runs_Scored
0,1,0
1,6,1
2,2,0
3,7,0
4,2,0
5,7,1
6,2,0
7,6,1
8,2,0
9,7,0


In [24]:
total_runs_scored_ps = """
SELECT Striker, Player_name, SUM(Runs_Scored) AS Total_Runs
FROM Ball_by_Ball b
JOIN Batsman_Scored s
  ON b.Match_ID = s.Match_ID
AND b.Over_ID = s.Over_ID
AND b.Ball_ID = s.Ball_ID
AND b.Innings_No = s.Innings_No
JOIN Player p
  ON b.Striker = p.Player_Id
GROUP BY Striker
ORDER BY Total_Runs DESC

"""
pd.read_sql_query(total_runs_scored_ps, conn)

,Striker,Player_Name,Total_Runs
0,21,SK Raina,4106
1,8,V Kohli,4105
2,57,RG Sharma,3874
3,40,G Gambhir,3634
4,162,CH Gayle,3447
...,...,...,...
429,222,S Ladda,0
430,215,YA Abdulla,0
431,193,C Nanda,0
432,145,Abdur Razzak,0


In [25]:
runs_per_match = """
SELECT striker, b.match_id, SUM(RUNS_SCORED) AS Runs_In_Match
FROM Ball_by_Ball b
JOIN Batsman_Scored s
  ON b.Match_ID = s.Match_ID
AND b.Over_ID = s.Over_ID
AND b.Ball_ID = s.Ball_ID
AND b.Innings_No = s.Innings_No
GROUP BY striker, b.match_id
"""
pd.read_sql_query(runs_per_match, conn)


,Striker,Match_Id,Runs_In_Match
0,1,335987,10
1,1,335991,14
2,1,335998,12
3,1,336002,4
4,1,336005,51
...,...,...,...
8611,431,981012,70
8612,432,980992,0
8613,433,981018,19
8614,433,981022,5


In [26]:
hundreds = """
SELECT Striker, SUM(CASE WHEN Runs_In_Match  >= 100 THEN 1 ELSE 0 END ) AS Hundreds, SUM(CASE WHEN Runs_In_Match  >= 50 AND Runs_In_Match < 100 THEN 1 ELSE 0 END ) AS Fifties
FROM (
SELECT striker, b.match_id, SUM(RUNS_SCORED) AS Runs_In_Match
FROM Ball_by_Ball b
JOIN Batsman_Scored s
  ON b.Match_ID = s.Match_ID
AND b.Over_ID = s.Over_ID
AND b.Ball_ID = s.Ball_ID
AND b.Innings_No = s.Innings_No
GROUP BY striker, b.match_id
) r
GROUP BY striker
ORDER BY Hundreds DESC
"""
pd.read_sql_query(hundreds, conn)

,striker,Hundreds,Fifties
0,162,5,21
1,8,4,27
2,110,3,22
3,187,2,32
4,185,2,13
...,...,...,...
429,6,0,11
430,5,0,0
431,4,0,5
432,3,0,0


In [32]:
wickets_per_match = """
SELECT Bowler, b.match_id, Count(*) AS Wickets_In_Match
FROM Ball_by_Ball b
JOIN Wicket_Taken w
  ON b.Match_ID = w.Match_ID
AND b.Over_ID = w.Over_ID
AND b.Ball_ID = w.Ball_ID
AND b.Innings_No = w.Innings_No
GROUP BY bowler, b.match_id
"""
pd.read_sql_query(wickets_per_match, conn)

,Bowler,Match_Id,Wickets_In_Match
0,1,335987,3
1,1,336015,1
2,1,336019,2
3,1,336032,1
4,1,392191,2
...,...,...,...
4047,462,981020,3
4048,463,980936,1
4049,463,980942,1
4050,463,980958,1


In [34]:
runs_conceded = """
SELECT Bowler, b.Match_id, SUM(RUNS_SCORED) AS Runs_Conceded
FROM Ball_by_Ball b
JOIN Batsman_Scored s
  ON b.Match_ID = s.Match_ID
AND b.Over_ID = s.Over_ID
AND b.Ball_ID = s.Ball_ID
AND b.Innings_No = s.Innings_No
GROUP BY Bowler, b.Match_id
"""
pd.read_sql_query(runs_conceded, conn)

,Bowler,Match_Id,Runs_Conceded
0,1,335987,20
1,1,336002,14
2,1,336005,14
3,1,336015,7
4,1,336019,25
...,...,...,...
6861,463,980964,51
6862,464,980962,22
6863,464,980970,31
6864,465,981002,13


In [41]:
bowling_per_match = """
SELECT r.Bowler, r.Match_id, r.Runs_Conceded, COALESCE(w.Wickets_In_Match, 0) AS Wickets
FROM (
SELECT Bowler, b.Match_id, SUM(RUNS_SCORED) AS Runs_Conceded
FROM Ball_by_Ball b
JOIN Batsman_Scored s
  ON b.Match_ID = s.Match_ID
AND b.Over_ID = s.Over_ID
AND b.Ball_ID = s.Ball_ID
AND b.Innings_No = s.Innings_No
GROUP BY Bowler, b.Match_id
) r
LEFT JOIN (
SELECT Bowler, b.match_id, Count(*) AS Wickets_In_Match
FROM Ball_by_Ball b
JOIN Wicket_Taken w
  ON b.Match_ID = w.Match_ID
AND b.Over_ID = w.Over_ID
AND b.Ball_ID = w.Ball_ID
AND b.Innings_No = w.Innings_No
GROUP BY bowler, b.match_id
) w
ON r.Bowler = w.Bowler
AND r.Match_id = w.Match_id
"""
pd.read_sql_query(bowling_per_match, conn)

,Bowler,Match_id,Runs_Conceded,Wickets
0,1,335987,20,3
1,1,336002,14,0
2,1,336005,14,0
3,1,336015,7,1
4,1,336019,25,2
...,...,...,...,...
6861,463,980964,51,0
6862,464,980962,22,0
6863,464,980970,31,3
6864,465,981002,13,0


In [42]:
best_bowling = """
WITH runs AS (
    SELECT b.Bowler, b.Match_Id, SUM(s.Runs_Scored) AS Runs_Conceded
    FROM Ball_by_Ball b
    JOIN Batsman_Scored s
      ON b.Match_Id = s.Match_Id AND b.Over_Id = s.Over_Id
     AND b.Ball_Id = s.Ball_Id AND b.Innings_No = s.Innings_No
    GROUP BY b.Bowler, b.Match_Id
),
wickets AS (
    SELECT b.Bowler, b.Match_Id, COUNT(*) AS Wickets
    FROM Ball_by_Ball b
    JOIN Wicket_Taken w
      ON b.Match_Id = w.Match_Id AND b.Over_Id = w.Over_Id
     AND b.Ball_Id = w.Ball_Id AND b.Innings_No = w.Innings_No
    GROUP BY b.Bowler, b.Match_Id
),
per_match AS (
    SELECT r.Bowler, r.Match_Id, r.Runs_Conceded,
           COALESCE(w.Wickets, 0) AS Wickets
    FROM runs r
    LEFT JOIN wickets w
      ON r.Bowler = w.Bowler AND r.Match_Id = w.Match_Id
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY Bowler
               ORDER BY Wickets DESC, Runs_Conceded ASC
           ) AS rn
    FROM per_match
)
SELECT p.Player_Name,
       k.Wickets || '/' || k.Runs_Conceded AS Best_Bowling
FROM ranked k
JOIN Player p ON p.Player_Id = k.Bowler
WHERE k.rn = 1
ORDER BY k.Wickets DESC, k.Runs_Conceded ASC
"""
pd.read_sql_query(best_bowling, conn)

,Player_Name,Best_Bowling
0,Sohail Tanvir,6/14
1,AD Russell,6/19
2,A Zampa,6/19
3,DJG Sammy,6/22
4,A Kumble,5/5
...,...,...
326,MB Parmar,0/32
327,RW Price,0/33
328,RR Bhatkal,0/35
329,Sunny Gupta,0/45


In [43]:
career = """
WITH bat_match AS (
    SELECT b.Striker AS Player_Id, b.Match_Id, SUM(s.Runs_Scored) AS Runs
    FROM Ball_by_Ball b
    JOIN Batsman_Scored s
      ON b.Match_Id = s.Match_Id AND b.Over_Id = s.Over_Id
     AND b.Ball_Id = s.Ball_Id AND b.Innings_No = s.Innings_No
    GROUP BY b.Striker, b.Match_Id
),
batting AS (
    SELECT Player_Id,
           SUM(Runs) AS Total_Runs,
           MAX(Runs) AS Highest_Score,
           SUM(CASE WHEN Runs >= 50 AND Runs < 100 THEN 1 ELSE 0 END) AS Fifties,
           SUM(CASE WHEN Runs >= 100 THEN 1 ELSE 0 END) AS Hundreds
    FROM bat_match
    GROUP BY Player_Id
),
bowl_runs AS (
    SELECT b.Bowler, b.Match_Id, SUM(s.Runs_Scored) AS Runs_Conceded
    FROM Ball_by_Ball b
    JOIN Batsman_Scored s
      ON b.Match_Id = s.Match_Id AND b.Over_Id = s.Over_Id
     AND b.Ball_Id = s.Ball_Id AND b.Innings_No = s.Innings_No
    GROUP BY b.Bowler, b.Match_Id
),
bowl_wickets AS (
    SELECT b.Bowler, b.Match_Id, COUNT(*) AS Wickets
    FROM Ball_by_Ball b
    JOIN Wicket_Taken w
      ON b.Match_Id = w.Match_Id AND b.Over_Id = w.Over_Id
     AND b.Ball_Id = w.Ball_Id AND b.Innings_No = w.Innings_No
    GROUP BY b.Bowler, b.Match_Id
),
bowl_match AS (
    SELECT r.Bowler, r.Match_Id, r.Runs_Conceded,
           COALESCE(w.Wickets, 0) AS Wickets
    FROM bowl_runs r
    LEFT JOIN bowl_wickets w
      ON r.Bowler = w.Bowler AND r.Match_Id = w.Match_Id
),
bowl_ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY Bowler
               ORDER BY Wickets DESC, Runs_Conceded ASC
           ) AS rn
    FROM bowl_match
),
bowling AS (
    SELECT Bowler AS Player_Id,
           SUM(Wickets) AS Total_Wickets,
           MAX(CASE WHEN rn = 1 THEN Wickets || '/' || Runs_Conceded END) AS Best_Bowling
    FROM bowl_ranked
    GROUP BY Bowler
),
matches AS (
    SELECT Player_Id, COUNT(DISTINCT Match_Id) AS Matches
    FROM Player_Match
    GROUP BY Player_Id
)
SELECT p.Player_Name,
       COALESCE(m.Matches, 0)         AS Matches,
       COALESCE(bat.Total_Runs, 0)    AS Total_Runs,
       COALESCE(bat.Highest_Score, 0) AS Highest_Score,
       COALESCE(bat.Fifties, 0)       AS Fifties,
       COALESCE(bat.Hundreds, 0)      AS Hundreds,
       COALESCE(bw.Total_Wickets, 0)  AS Total_Wickets,
       COALESCE(bw.Best_Bowling, '-') AS Best_Bowling
FROM Player p
LEFT JOIN matches m   ON m.Player_Id = p.Player_Id
LEFT JOIN batting bat ON bat.Player_Id = p.Player_Id
LEFT JOIN bowling bw  ON bw.Player_Id = p.Player_Id
ORDER BY Total_Runs DESC
"""
pd.read_sql_query(career, conn)

,Player_Name,Matches,Total_Runs,Highest_Score,Fifties,Hundreds,Total_Wickets,Best_Bowling
0,SK Raina,146,4106,100,28,1,29,2/0
1,V Kohli,138,4105,113,27,4,5,2/23
2,RG Sharma,142,3874,109,29,1,16,4/6
3,G Gambhir,132,3634,93,31,0,0,-
4,CH Gayle,91,3447,175,21,5,19,3/21
...,...,...,...,...,...,...,...,...
464,DL Chahar,2,0,0,0,0,0,0/13
465,P Dharmani,1,0,0,0,0,0,-
466,RV Pawar,1,0,0,0,0,0,-
467,KH Devdhar,1,0,0,0,0,0,-
